# Boca Juniors 2026 | Comparación de ciclos

El análisis abarca los 32 partidos de zonas que sumaron para la tabla anual.
Los balances se calculan desde los marcadores. La asignación de los
partidos a cada ciclo está documentada en [fuentes](../docs/fuentes.md).

In [1]:
from pathlib import Path

import pandas as pd

from lpf_analytics.resultados import resumen_equipo

actual = Path.cwd()
raiz = next(
    carpeta
    for carpeta in (actual, *actual.parents)
    if (carpeta / "pyproject.toml").is_file()
)
equipo = "Boca Juniors"
partidos = pd.read_csv(
    raiz / "data/raw/LPF_2026_partidos.csv", encoding="utf-8-sig"
)
es_boca = partidos[["local", "visitante"]].eq(equipo).any(axis=1)
boca = partidos.loc[es_boca].copy()
boca["condicion"] = boca["local"].eq(equipo).map({
    True: "Local",
    False: "Visitante",
})


def balance_por(claves):
    filas = []
    for valores, grupo in boca.groupby(claves, dropna=False):
        balance = resumen_equipo(grupo, equipo)
        filas.append({**dict(zip(claves, valores)), **balance})

    tabla = pd.DataFrame(filas)
    tabla["DG"] = tabla["GF"] - tabla["GC"]
    tabla["PTS/PJ"] = (tabla["PTS"] / tabla["PJ"]).round(2)
    return tabla

## Balance por ciclo

In [2]:
balance_por(["torneo", "dt_boca"])

,torneo,dt_boca,PJ,PG,PE,PP,PTS,GF,GC,DG,PTS/PJ
0,Apertura,Claudio Úbeda,16,8,6,2,30,22,9,13,1.88
1,Clausura,Rodolfo Arruabarrena,16,9,6,1,33,24,14,10,2.06


## Balance por localía

In [3]:
balance_por(["torneo", "dt_boca", "condicion"])

,torneo,dt_boca,condicion,PJ,PG,PE,PP,PTS,GF,GC,DG,PTS/PJ
0,Apertura,Claudio Úbeda,Local,8,3,5,0,14,8,3,5,1.75
1,Apertura,Claudio Úbeda,Visitante,8,5,1,2,16,14,6,8,2.00
2,Clausura,Rodolfo Arruabarrena,Local,8,7,1,0,22,13,3,10,2.75
3,Clausura,Rodolfo Arruabarrena,Visitante,8,2,5,1,11,11,11,0,1.38


## Qué cambió entre los ciclos

Boca sumó 30 puntos con Úbeda y 33 con Arruabarrena en 16 partidos por ciclo. De local pasó de 14 a 22 puntos y de tres a siete victorias en ocho partidos. De visitante pasó de 16 a 11 puntos.

En el Clausura marcó dos goles más y recibió cinco más. Todo el aumento de goles recibidos se dio de visitante, donde pasó de 6 a 11. De local recibió 3 en cada ciclo.

La comparación es descriptiva. Cada ciclo coincide con un torneo distinto, por lo que el cambio de entrenador y el cambio de torneo se observan juntos.

In [4]:
from IPython.display import display

from lpf_analytics.remates import cargar_xg_boca

carpeta_remates = raiz / "data/raw"
archivos_remates = [
    "strikerapi_boca_partidos_2026.json",
    "strikerapi_boca_remates_favor_2026.json",
    "strikerapi_boca_remates_contra_2026.json",
]
faltantes = [
    nombre for nombre in archivos_remates
    if not (carpeta_remates / nombre).is_file()
]

if faltantes:
    print("Para calcular xG y xGA, guardá estos archivos en data/raw.")
    print("\n".join(faltantes))
else:
    remates_por_partido = cargar_xg_boca(carpeta_remates)
    xg_por_torneo = remates_por_partido.groupby("torneo", as_index=False).agg(
        PJ_con_xG=("xG_registrado", "size"),
        xG_registrado=("xG_registrado", "sum"),
        xGA_registrado=("xGA_registrado", "sum"),
    )
    comparacion_xg = balance_por(["torneo", "dt_boca"]).merge(
        xg_por_torneo, on="torneo", how="left", validate="one_to_one"
    )
    if not comparacion_xg["PJ"].eq(comparacion_xg["PJ_con_xG"]).all():
        raise ValueError("La cantidad de partidos con xG difiere del balance del ciclo.")

    columnas = ["torneo", "dt_boca", "PJ", "GF", "xG_registrado", "GC", "xGA_registrado"]
    display(comparacion_xg[columnas].round(2))

,torneo,dt_boca,PJ,GF,xG_registrado,GC,xGA_registrado
0,Apertura,Claudio Úbeda,16,22,20.58,9,12.40
1,Clausura,Rodolfo Arruabarrena,16,24,24.82,14,14.63


### Qué cambió en el xG registrado

En 16 partidos por ciclo, el xG registrado de Boca pasó de **20,58 a 24,82**. El xGA registrado también subió, de **12,40 a 14,63**. En el Clausura, tanto Boca como sus rivales acumularon más xG en los remates disponibles.

StrikerAPI devolvió remates con xG de ambos equipos en los 32 partidos de zonas. El detalle puede ser parcial dentro de cada encuentro. Ante Deportivo Riestra en el Apertura, por ejemplo, 22 de los 25 remates informados tenían coordenadas y xG.

La comparación es descriptiva. Cada ciclo coincide con un torneo distinto. [Cobertura y método de cálculo](../docs/fuentes.md#cobertura-de-strikerapi-para-boca)

In [5]:
from lpf_analytics.remates import cargar_xpts_boca

if faltantes:
    print("Para calcular xPTS, guardá los JSON indicados en la sección de xG.")
else:
    xpts_partidos = cargar_xpts_boca(carpeta_remates)
    xpts_por_torneo = xpts_partidos.groupby("torneo", as_index=False)["xPTS"].sum()

    comparacion_xpts = comparacion_xg[["torneo", "dt_boca", "PJ", "PTS"]].merge(
        xpts_por_torneo, on="torneo", validate="one_to_one"
    )
    comparacion_xpts["PTS_menos_xPTS"] = (
        comparacion_xpts["PTS"] - comparacion_xpts["xPTS"]
    )

    display(comparacion_xpts.round(2))
    print(f"xPTS anual: {xpts_partidos['xPTS'].sum():.2f}")

,torneo,dt_boca,PJ,PTS,xPTS,PTS_menos_xPTS
0,Apertura,Claudio Úbeda,16,30,28.21,1.79
1,Clausura,Rodolfo Arruabarrena,16,33,29.08,3.92


xPTS anual: 57.30


### Puntos esperados por ciclo

Boca sumó **63 puntos reales** frente a **57,30 xPTS** estimados. La diferencia fue mayor en el Clausura, con **+3,92 puntos**, frente a **+1,79** en el Apertura.

El modelo usa el xG de cada remate registrado como probabilidad de gol y supone independencia entre remates. Calcula las probabilidades de victoria, empate y derrota y asigna `3 × P(victoria) + P(empate)`. La estimación describe las ocasiones registradas después de cada partido. [Método de cálculo](https://dtai.cs.kuleuven.be/sports/blog/expected-goals-and-the-monte-carlo-trap/)

Los totales se suman antes de redondear.

In [6]:
import json

from lpf_analytics.scouting import resumir_remates_jugadores

ruta_partidos = carpeta_remates / "strikerapi_boca_partidos_2026.json"
ruta_favor = carpeta_remates / "strikerapi_boca_remates_favor_2026.json"

if ruta_partidos.is_file() and ruta_favor.is_file():
    partidos_api = json.loads(ruta_partidos.read_text(encoding="utf-8-sig"))["games"]
    remates_api = json.loads(ruta_favor.read_text(encoding="utf-8-sig"))["shots"]

    ids_zonas = {partido["game_id"] for partido in partidos_api}
    detalle = pd.DataFrame(remates_api)
    remates_zonas = detalle.loc[detalle["game_id"].isin(ids_zonas)]
    ranking_remates = resumir_remates_jugadores(remates_zonas)

    columnas = [
        "Jugador", "Remates", "Goles", "Penales", "xG_registrado",
        "Remates_sin_penal", "npxG_registrado", "npxG_por_remate",
    ]
    top_remates = ranking_remates[columnas].head(10).round(2)
    display(top_remates.reset_index(drop=True))
else:
    print("Para calcular el ranking, guardá los JSON de partidos y remates a favor.")

,Jugador,Remates,Goles,Penales,xG_registrado,Remates_sin_penal,npxG_registrado,npxG_por_remate
0,Miguel Merentiel,54,10,0,8.46,54,8.46,0.16
1,Santiago Ascacibar,42,5,0,5.80,42,5.80,0.14
2,Adam Bareiro,18,3,0,3.65,18,3.65,0.20
3,Leonel Flores,36,3,0,3.07,36,3.07,0.09
4,Alan Velasco,31,4,0,2.23,31,2.23,0.07
5,Tomás Aranda,35,1,0,2.20,35,2.20,0.06
6,Lautaro Di Lollo,19,3,0,2.08,19,2.08,0.11
7,Enner Valencia,14,3,0,2.05,14,2.05,0.15
8,Tomás Belmonte,11,1,0,1.54,11,1.54,0.14
9,Milton Gimenez,12,3,1,2.03,11,1.29,0.12


### Perfil de remate por jugador

El ranking ordena a los diez jugadores con más npxG registrado. El npxG suma el xG de los remates sin penal. StrikerAPI identificó al jugador en los 434 remates de zonas, que incluyen 3 penales.

Miguel Merentiel lideró con **8,46 npxG en 54 remates sin penal**. Adam Bareiro promedió **0,20 xG por remate sin penal** en **18 intentos**, frente a **0,16** de Merentiel. El volumen de remates da contexto a esos promedios.

Los indicadores describen los remates registrados. Para calcular tasas por 90 minutos hacen falta los minutos jugados por cada futbolista. [Cobertura de la fuente](../docs/fuentes.md#cobertura-de-strikerapi-para-boca)